# 01 Ingest, split, index

Fetches the allowlisted RBI documents in `configs/rbi_sources.yaml` into `corpus/{doc_id}.jsonl` (one row per
paragraph), splits documents into train/dev/test by topic (`data/splits.json`), and builds the hybrid BM25 +
embedding index under `index/<version>/`.
Manifests: `corpus/state.json` and `index/<version>/state.json` (state `INGESTED`, with an input hash; a re-run
with unchanged inputs skips the work). If any document fails, the corpus manifest is not written so the stage reruns.

**Needs:** CPU only. PDFs on rbidocs.rbi.org.in show a CAPTCHA to scripts: open each `url` that has a `file` in
`configs/rbi_sources.yaml` in a browser and save it at that path first. HTML pages fetch automatically.

In [ ]:
import json, os, sys
from collections import Counter
from pathlib import Path

# Run from the repo root so relative paths match the README runbook.
ROOT = Path.cwd()
while not (ROOT / "core").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PY = sys.executable  # `!{PY} -m ...` runs the CLI with this kernel's interpreter
print("repo root:", ROOT)

In [ ]:
SOURCES = "configs/rbi_sources.yaml"
CORPUS = "corpus/"
SPLITS = "data/splits.json"
INDEX_ROOT = "index/"
INDEX_VERSION = "rbi-2026-11-10"
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

In [ ]:
from pipeline.ingest import ingest

rc = ingest(Path(SOURCES), Path(CORPUS))
assert rc == 0, "some documents failed (see stderr); fix them and re-run this cell"

In [ ]:
!{PY} -m pipeline.split {CORPUS} {SPLITS}

In [ ]:
from retrieval.build_index import build

INDEX = build(Path(CORPUS), Path(INDEX_ROOT), INDEX_VERSION, EMBEDDING_MODEL)

## Summary

In [ ]:
def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.open(encoding="utf-8")] if path.exists() else []


files = sorted(Path(CORPUS).glob("*.jsonl"))
print(f"corpus: {len(files)} documents, {sum(len(read_jsonl(f)) for f in files)} paragraphs")
print("corpus manifest:", json.loads((Path(CORPUS) / "state.json").read_text()))
splits = json.loads(Path(SPLITS).read_text())
print("docs per split:", {k: len(v) for k, v in splits.items()})
index_dir = Path(INDEX_ROOT) / INDEX_VERSION
print("index meta:", json.loads((index_dir / "meta.json").read_text()))
for p in read_jsonl(index_dir / "passages.jsonl")[:3]:
    print(f"- {p['pid']}: {p['text'][:120]}")